In [3]:
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
from argparse import Namespace
import torch
import requests
import numpy as np
import cv2
from PIL import Image
import utils_model_qwen as utils_model
import utils_gradio_qwen as utils_gradio
import utils_attn
device_map = "auto"

In [4]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map=device_map
)

Loading checkpoint shards: 100%|████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.03steps/s]


In [5]:
processor = AutoProcessor.from_pretrained(model_name_or_path)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [6]:
utils_gradio.processor = processor
utils_gradio.model = model

In [7]:
prompt = "What animal is this and what is on the left of it?"
image_file = "http://images.cocodataset.org/val2017/000000039769.jpg"
raw_image = Image.open(requests.get(image_file, stream=True).raw)

image_process_mode = "Default"
state, _, _, _ = utils_gradio.add_text(None, prompt, raw_image, image_process_mode)

In [11]:
temperature = 0.1
top_p = 0.7
max_new_tokens = 20

state, _ = utils_gradio.lvlm_bot(state, temperature, top_p, max_new_tokens)

recovered_image = state.recovered_image
fig = plt.figure()
plt.imshow(recovered_image)
role, message = state.messages[-1]
print(f'{role}: {message}')

IndexError: index 0 is out of bounds for dimension 0 with size 0

In [8]:
prompt = state.prompt
prompt_len = state.prompt_len
image = state.image

inputs = processor(image, prompt, return_tensors="pt").to(model.device)
input_ids = inputs.input_ids

In [9]:
prompt

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<image>\nWhat animal is this and what is on the left of it?<|im_end|>\n<|im_start|>assistant\n'

In [10]:
prompt_len

166

In [11]:
inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     27,   1805,    397,   3838,
           9864,    374,    419,    323,   1128,    374,    389,    279,   2115,
            315,    432,     30, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0'), 'pixel_values': tensor([[ 0.2515,  0.3099,  0.3391,  ..., -0.6270, -0.3995, -0.4990],
        [ 0.5143,  0.2807,  0.5581,  ..., -0.2857, -0.4137, -0.2573],
        [-0.0113,  0.0909, -0.0842,  ..., -1.0963, -1.0252, -0.9683],
        ...,
        [ 1.7114,  1.6238,  1.6238,  ...,  1.1505,  1.0652,  1.0225],
        [ 1.4486,  1.5800,  1.5216,  ..., -0.3426, -0.2146,  0.2688],
        [ 1.6530,  1.6676,  1.5508,  ..., -1.0678, -0.8545, -0.8830]],
       device='cuda:0'), 'image_

In [16]:
len(input_ids[0])

35

In [17]:
torch.where(input_ids==model.config.image_token_index)

(tensor([], device='cuda:0', dtype=torch.int64),
 tensor([], device='cuda:0', dtype=torch.int64))

In [18]:
generated_ids = model.generate(**inputs, max_new_tokens=128)

ValueError: Image features and image tokens do not match: tokens: 0, features 391

In [19]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

In [20]:
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [21]:
text

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Describe this image.<|im_end|>\n<|im_start|>assistant\n'

In [23]:
from qwen_vl_utils import process_vision_info

image_inputs, video_inputs = process_vision_info(messages)

In [28]:
type(image_inputs[0])

PIL.Image.Image

In [30]:
type(raw_image)

PIL.JpegImagePlugin.JpegImageFile

In [31]:
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

In [33]:
inputs

{'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'pixel_values': tensor([[ 0.8501,  0.8501,  0.8647,  ...,  1.3922,  1.3922,  1.3922],
        [ 0.9376,  0.9376,  0.9376,  ...,  1.4491,  1.4491,  1.4491],
        [ 0.9084,  0.9376,  0.9376,  ...,  1.4065,  1.4207,  1.4207],
        ...,
        [-0.1280, -0.1280, -0.1426,  ..., -0.2431, -0.2715, -0.3000],
        [-0.3324, -0.3324, -0.3032,  ..., -0.3000, -0.2715, -0.2857],
        [-0.3762, -0.4054, -0.4054,  ..., -0.4279, -0.4422, -0.4564]],
       device='cuda:0'), 'image_grid_thw': tensor([[  1,  98, 146]], device='cuda:0')}

In [37]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw'])

In [32]:
generated_ids = model.generate(**inputs, max_new_tokens=128)